# Whisper paraphasia detection — Scripts-Fridriksson LOSO on Kaggle (2× T4)

Data comes from the private Kaggle dataset `evandiewald/aphasia-scripts-fridriksson` (fetched with kagglehub, no need to attach it).
Needs the `WANDB_API_KEY` Kaggle secret and internet on.

Two folds train at once, one per GPU, in a background process that keeps running if this notebook disconnects.
Each finished fold is uploaded as a W&B artifact (`<RUN_NAME>-fold_<spk>`); re-running the launch cell in a new session skips uploaded folds.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [ ]:
import os, subprocess

REPO_DIR = '/kaggle/working/aphasia-modeling'
RUN_NAME = 'scripts-small-taginit'
CKPT = f'/root/checkpoints/{RUN_NAME}'  # / has ~1 TB; /kaggle/working is capped at 20 GB
RESULTS = f'/kaggle/working/results/{RUN_NAME}'
DATA_PATH = 'datasets/Scripts/scripts_fridriksson.json'
os.makedirs(CKPT, exist_ok=True)

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/evandiewald/aphasia-modeling.git {REPO_DIR}
!cd {REPO_DIR} && git log --oneline -1

In [ ]:
!cd {REPO_DIR} && pip install -e . -q

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

In [ ]:
# Link the dataset into the repo at the paths the JSON references
import json, kagglehub
src = kagglehub.dataset_download('evandiewald/aphasia-scripts-fridriksson')
audio_dir = f'{REPO_DIR}/datasets/Scripts/Fridriksson/audio'
os.makedirs(os.path.dirname(audio_dir), exist_ok=True)
if not os.path.islink(audio_dir):
    os.symlink(f'{src}/audio', audio_dir)
!cp {src}/scripts_fridriksson.json {REPO_DIR}/{DATA_PATH}

records = json.load(open(f'{REPO_DIR}/{DATA_PATH}'))
missing = {r['audio_path'] for r in records if not os.path.exists(f"{REPO_DIR}/{r['audio_path']}")}
print(f"{len(records)} utterances, {len({r['speaker_id'] for r in records})} speakers, {len(missing)} missing audio files")
assert not missing, sorted(missing)[:5]

In [ ]:
# Launch all folds in the background, one per GPU (T4: fp16, batch 4 x accum 4 = effective 16)
TAG_CLASSES = 'pns'  # 'pn' to leave [s] to Stage 2
if subprocess.run(['pgrep', '-f', 'run_loso_parallel'], capture_output=True).returncode == 0:
    print('Runner already running')
else:
    cmd = (f'cd {REPO_DIR} && nohup python scripts/run_loso_parallel.py --gpus 0,1 '
           f'--data_path {DATA_PATH} --output_dir {CKPT} --run_name {RUN_NAME} --wandb_artifacts -- '
           f'--tag_classes {TAG_CLASSES} --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb '
           f'>> {CKPT}/runner.log 2>&1 &')
    subprocess.run(cmd, shell=True, start_new_session=True)
    print(f'Started; runner log: {CKPT}/runner.log')

In [ ]:
# Progress: re-run any time
import re, glob
print(open(f'{CKPT}/runner.log').read()[-2000:])
for log in sorted(glob.glob(f'{CKPT}/logs/*.log')):
    evals = re.findall(r"\{'eval_loss'.*?'epoch': '[\d.]+'\}", open(log).read())
    print(os.path.basename(log), evals[-1] if evals else '(no eval yet)')
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

In [ ]:
# After 'All folds complete': fetch every fold and run pooled LOSO evaluation
!cd {REPO_DIR} && python scripts/run_loso_parallel.py --download --data_path {DATA_PATH} --output_dir {CKPT} --run_name {RUN_NAME}
!cd {REPO_DIR} && python scripts/evaluate_model.py --model_dir {CKPT} --data_path {DATA_PATH} \
    --loso --output_dir {RESULTS} --batch_size 16

In [2]:
import os, subprocess, json
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
print('secret ok')
REPO_DIR = '/kaggle/working/aphasia-modeling'
RUN_NAME = 'scripts-small-taginit'
CKPT = f'/root/checkpoints/{RUN_NAME}'
DATA_PATH = 'datasets/Scripts/scripts_fridriksson.json'
os.makedirs(CKPT, exist_ok=True)
sh = lambda c: print(subprocess.run(c + " 2>&1", shell=True, capture_output=True, text=True, executable='/bin/bash').stdout[-3000:])
sh(f"if [ -d {REPO_DIR} ]; then cd {REPO_DIR} && git pull; else git clone -q https://github.com/evandiewald/aphasia-modeling.git {REPO_DIR}; fi; cd {REPO_DIR} && git log --oneline -1 && ls scripts")

secret ok
7961d92 update
evaluate_model.py
run_loso_parallel.py
train.py



In [3]:
sh(f"cd {REPO_DIR} && pip install -e . -q 2>&1 | tail -5; python -c 'import aphasia_modeling, torch, transformers; print(\"ok\", torch.__version__, transformers.__version__)'")

google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
notebook 6.5.7 requires jupyter-client<8,>=5.3.4, but you have jupyter-client 8.10.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter_client<8.0,>=5.2.0, but you have jupyter-client 8.10.0 which is incompatible.
ok 2.11.0+cu128 5.16.1



In [4]:
import kagglehub
src = kagglehub.dataset_download('evandiewald/aphasia-scripts-fridriksson')
audio_dir = f'{REPO_DIR}/datasets/Scripts/Fridriksson/audio'
os.makedirs(os.path.dirname(audio_dir), exist_ok=True)
if not os.path.islink(audio_dir):
    os.symlink(f'{src}/audio', audio_dir)
sh(f"cp {src}/scripts_fridriksson.json {REPO_DIR}/{DATA_PATH}")
records = json.load(open(f'{REPO_DIR}/{DATA_PATH}'))
missing = {r['audio_path'] for r in records if not os.path.exists(f"{REPO_DIR}/{r['audio_path']}")}
print(f"{len(records)} utterances, {len({r['speaker_id'] for r in records})} speakers, {len(missing)} missing audio files")


985 utterances, 12 speakers, 0 missing audio files


In [5]:
sh("""python -c "import wandb; a=wandb.Api(); print('entity:', a.default_entity)" """)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
entity: evandiewald-aws



In [6]:
TAG_CLASSES = 'pns'
if subprocess.run(['pgrep', '-f', 'run_loso_parallel'], capture_output=True).returncode == 0:
    print('Runner already running')
else:
    cmd = (f'cd {REPO_DIR} && nohup python scripts/run_loso_parallel.py --gpus 0,1 '
           f'--data_path {DATA_PATH} --output_dir {CKPT} --run_name {RUN_NAME} --wandb_artifacts -- '
           f'--tag_classes {TAG_CLASSES} --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb '
           f'>> {CKPT}/runner.log 2>&1 &')
    subprocess.run(cmd, shell=True, start_new_session=True)
    print('Started')
import time; time.sleep(90)
sh(f"cat {CKPT}/runner.log; tail -c 1500 {CKPT}/logs/fold_P1.log; nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv")

Started
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
[P1] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P1.log
[P10] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P10.log
vandiewald (evandiewald-aws) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: setting up run ps3ctlai
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/aphasia-modeling/wandb/run-20261003_003055-ps3ctlai
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scripts-small-taginit-P1
wandb: ⭐️ View project at https://wandb.ai/evandiewald-aws/talking-points
wandb: 🚀 View run at https://wandb.ai/evandiewald-aws/talking-points/runs/ps3ctlai

  1%|          | 10/1000 [00:43<33:49,  2.05s/it]
                                                 

  2%|▏         | 15/1000 [00:52<29:21, 

In [7]:
import re, glob
time.sleep(240)
def progress():
    for log in sorted(glob.glob(f'{CKPT}/logs/*.log')):
        txt = open(log).read()
        evals = re.findall(r"\{'eval_loss'.*?'epoch': '[\d.]+'\}", txt)
        print(os.path.basename(log))
        for e in evals: print('  ', re.sub(r"'eval_(runtime|samples_per_second|steps_per_second)': '[^']*', ", '', e))
        if 'Traceback' in txt: print(txt[txt.index('Traceback'):][:1500])
progress()

fold_P1.log
   {'eval_loss': '2.202', 'eval_ref_p': 100, 'eval_pred_p': 117, 'eval_recall_p': '0.28', 'eval_ref_n': 56, 'eval_pred_n': 1, 'eval_recall_n': '0', 'eval_ref_s': 27, 'eval_pred_s': 1, 'eval_recall_s': '0', 'eval_tag_class_acc': '0.5301', 'epoch': '1'}
   {'eval_loss': '1.396', 'eval_ref_p': 100, 'eval_pred_p': 81, 'eval_recall_p': '0.45', 'eval_ref_n': 56, 'eval_pred_n': 61, 'eval_recall_n': '0.5536', 'eval_ref_s': 27, 'eval_pred_s': 29, 'eval_recall_s': '0.1852', 'eval_tag_class_acc': '0.6175', 'epoch': '2'}
fold_P10.log
   {'eval_loss': '2.287', 'eval_ref_p': 91, 'eval_pred_p': 99, 'eval_recall_p': '0.2747', 'eval_ref_n': 57, 'eval_pred_n': 3, 'eval_recall_n': '0', 'eval_ref_s': 26, 'eval_pred_s': 1, 'eval_recall_s': '0', 'eval_tag_class_acc': '0.4943', 'epoch': '1'}
   {'eval_loss': '1.427', 'eval_ref_p': 91, 'eval_pred_p': 118, 'eval_recall_p': '0.4945', 'eval_ref_n': 57, 'eval_pred_n': 72, 'eval_recall_n': '0.5263', 'eval_ref_s': 26, 'eval_pred_s': 29, 'eval_recall_s':

In [8]:
sh(f"ps aux | grep -E 'run_loso|train.py' | grep -v grep | cut -c1-200; echo ---; tail -20 {CKPT}/runner.log; echo ---; nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv; ls {CKPT} {CKPT}/fold_P1 2>&1")
progress()

root         294  0.3  0.4 799212 148020 ?       Sl   00:30   0:01 python3 scripts/run_loso_parallel.py --gpus 0,1 --data_path datasets/Scripts/scripts_fridriksson.json --output_dir /root/checkpoints/
root         320 80.7  8.1 25108060 2666224 ?    Rl   00:30   6:23 /usr/bin/python3 /kaggle/working/aphasia-modeling/scripts/train.py --data_path datasets/Scripts/scripts_fridriksson.json --output_dir
root         321 80.1  8.3 25636656 2753168 ?    Rl   00:30   6:21 /usr/bin/python3 /kaggle/working/aphasia-modeling/scripts/train.py --data_path datasets/Scripts/scripts_fridriksson.json --output_dir
root         762 12.6  6.8 25238896 2254304 ?    Sl   00:37   0:09 /usr/bin/python3 /kaggle/working/aphasia-modeling/scripts/train.py --data_path datasets/Scripts/scripts_fridriksson.json --output_dir
root         763 12.5  6.8 25238908 2254208 ?    Sl   00:37   0:09 /usr/bin/python3 /kaggle/working/aphasia-modeling/scripts/train.py --data_path datasets/Scripts/scripts_fridriksson.json --output

In [9]:
def status():
    sh(f"tail -8 {CKPT}/runner.log; echo ---; pgrep -fc 'scripts/train.py'; nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader")
    for log in sorted(glob.glob(f'{CKPT}/logs/*.log')):
        txt = open(log).read()
        evals = re.findall(r"\{'eval_loss'.*?'epoch': '[\d.]+'\}", txt)
        last = evals[-1] if evals else ''
        g = lambda k: (re.search(rf"'eval_{k}': '?([\d.]+)", last) or [None, '?'])[1]
        print(f"{os.path.basename(log):16s} epochs={len(evals):2d} loss={g('loss')} pred/ref p={g('pred_p')}/{g('ref_p')} n={g('pred_n')}/{g('ref_n')} s={g('pred_s')}/{g('ref_s')} tagacc={g('tag_class_acc')}"
              + ("  FOLD COMPLETE" if 'Fold complete' in txt else "") + ("  TRACEBACK" if 'Traceback' in txt else ""))
status()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
[P1] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P1.log
[P10] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P10.log
---
7
0, 97 %, 7555 MiB
1, 97 %, 7567 MiB

fold_P1.log      epochs= 4 loss=1.274 pred/ref p=119/100 n=43/56 s=12/27 tagacc=0.6175
fold_P10.log     epochs= 4 loss=1.294 pred/ref p=104/91 n=44/57 s=21/26 tagacc=0.6092


In [10]:
status()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
[P1] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P1.log
[P10] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P10.log
[P1] done
[P11] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P11.log
[P10] done
[P12] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P12.log
---
7
0, 100 %, 7567 MiB
1, 100 %, 7579 MiB

fold_P1.log      epochs= 8 loss=1.359 pred/ref p=133/100 n=29/56 s=22/27 tagacc=0.623  FOLD COMPLETE
fold_P10.log     epochs= 9 loss=1.478 pred/ref p=100/91 n=64/57 s=23/26 tagacc=0.6149  FOLD COMPLETE
fold_P11.log     epochs= 6 loss=1.363 pred/ref p=126/105 n=46/60 s=29/27 tagacc=0.6302
fold_P12.log     epochs= 5 loss=1.256 pred/ref p=65/89 n=71/50 s=14/20 tagacc=0.5975


In [11]:
sh("""python -c "
import wandb; a=wandb.Api()
for s in ['P1','P10']:
    art=a.artifact(f'{a.default_entity}/talking-points/scripts-small-taginit-fold_{s}:latest'); print(s, art.version, round(art.size/1e6), 'MB', sorted(f.name for f in art.files())[:8])
" 2>&1 | grep -v Loaded""")
sh(f"cat {CKPT}/fold_P1/train_metrics.json | tr -d '\\n'; echo; cat {CKPT}/fold_P10/train_metrics.json | tr -d '\\n'")

P1 v0 971 MB ['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'train_metrics.json', 'training_args.bin']
P10 v0 971 MB ['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'train_metrics.json', 'training_args.bin']

{  "train_runtime": 899.9775,  "train_samples_per_second": 17.512,  "train_steps_per_second": 1.111,  "total_flos": 1.81924236361728e+18,  "train_loss": 3.610258446931839,  "epoch": 8.0,  "test_speaker": "P1",  "train_size": 788,  "dev_size": 89,  "test_size": 91,  "best_eval_loss": 1.254693865776062}
{  "train_runtime": 1049.0806,  "train_samples_per_second": 15.118,  "train_steps_per_second": 0.953,  "total_flos": 2.05963400208384e+18,  "train_loss": 3.373892382250892,  "epoch": 9.0,  "test_speaker": "P10",  "train_size": 793,  "dev_size": 90,  "test_size": 92,  "best_eval_loss": 1.294057011604309}


In [12]:
sh(f"cd {REPO_DIR} && CUDA_VISIBLE_DEVICES=0 python scripts/evaluate_model.py --model_dir {CKPT}/fold_P1 --data_path {DATA_PATH} --test_speaker P1 --output_dir /kaggle/working/results/early_P1 --batch_size 8 2>&1 | grep -v 'it/s' | grep -A16 'Pooled'")
rows = json.load(open('/kaggle/working/results/early_P1/predictions.json'))
from collections import Counter
c = Counter()
for r in rows:
    for k in ['[p]','[n]','[s]']:
        c['ref'+k] += r['reference'].split().count(k); c['hyp'+k] += r['hypothesis'].split().count(k)
print(dict(c))

Pooled over 1 fold(s), 91 utterances
WER:       0.6515
AWER:      0.7300
AWER-PD:   0.6768
TD-binary: 1.0462
TD-[p]:    1.1568
TD-[n]:    1.3175
TD-[s]:    1.6004
TD-all:    4.0746
F1-[p]:    0.7500
F1-[n]:    0.4595
F1-[s]:    0.4103
Recall-[p]: 0.8136
Recall-[n]: 0.3269
Recall-[s]: 0.6400
Utterances: 91

{'ref[p]': 96, 'hyp[p]': 165, 'ref[n]': 81, 'hyp[n]': 30, 'ref[s]': 34, 'hyp[s]': 145}


In [13]:
import random; random.seed(0)
for r in random.sample(rows, 8): print('REF:', r['reference']); print('HYP:', r['hypothesis']); print()

REF: is easy to use
HYP: then easy [s] is easy [s] is

REF: during the fall it is cool and the leaves change eschuhl [n]
HYP: during the fall it is cool and to leave [s] [s]

REF: pan pan the pambuhld [p] the egg butter and heat heat them up up them over medium heat
HYP: and i [s] [s] [s] [s] [s] [s] [s] [s]

REF: during the fall this [s] is tool [p] and the leaves turn ttuhz [n]
HYP: during the fall this is [s] [s] [s] [s] [s] [s] [s] [s]

REF: fees [p] speak directing to me and din [n] me time for myoonikayt [n]
HYP: please me to write them to me and then me tan [p] for yoonuhkayt [n]

REF: very [s] is ee [n] to use
HYP: feuhli [p] is [s] to use

REF: aphasia fekts [p] my language not my ditikuhlt [n]
HYP: i [s] do [s] i [s] language i [s] tithoo [p]

REF: he [p] me find the words say theem [p] eluhi [n]
HYP: they find the [s] words [s] say [p] the [s] entity



In [14]:
test_code = r'''
import sys, json, torch, librosa
sys.path.insert(0, "src")
from transformers import LogitsProcessor, LogitsProcessorList
from aphasia_modeling.data.dataset import AphasiaBankDataset
from aphasia_modeling.model.inference import ParaphasiaPredictor, SAMPLING_RATE
from aphasia_modeling.model.tokenizer import get_paraphasia_token_ids
from aphasia_modeling.evaluation.metrics import compute_all_metrics
from collections import Counter

class NoTagAfterTag(LogitsProcessor):
    def __init__(self, tag_ids): self.tag_ids = torch.tensor(tag_ids)
    def __call__(self, input_ids, scores):
        t = self.tag_ids.to(scores.device)
        after_tag = torch.isin(input_ids[:, -1], t)
        scores[after_tag.nonzero(as_tuple=True)[0].unsqueeze(1), t] = -float("inf")
        return scores

ds = AphasiaBankDataset.load("datasets/Scripts/scripts_fridriksson.json")
_, _, test = ds.loso_split("P1")
pred = ParaphasiaPredictor("/root/checkpoints/scripts-small-taginit/fold_P1", device="cuda")
pred._gen_kwargs["logits_processor"] = LogitsProcessorList([NoTagAfterTag(list(get_paraphasia_token_ids(pred.tokenizer).values()))])
rows = []
for i in range(0, len(test), 8):
    b = test[i:i+8]
    auds = [librosa.load(u.audio_path, sr=SAMPLING_RATE, offset=u.start_time, duration=u.end_time-u.start_time)[0] if u.end_time else librosa.load(u.audio_path, sr=SAMPLING_RATE)[0] for u in b]
    for u, h in zip(b, pred.predict_batch(auds)):
        rows.append({"reference": " ".join(w + (f" [{l}]" if l != "c" else "") for w, l in zip(u.words, u.labels)), "hypothesis": h})
m = compute_all_metrics([r["reference"].split() for r in rows], [r["hypothesis"].split() for r in rows])
print(m)
c = Counter()
for r in rows:
    for k in ["[p]","[n]","[s]"]:
        c["ref"+k] += r["reference"].split().count(k); c["hyp"+k] += r["hypothesis"].split().count(k)
print(dict(c))
json.dump(rows, open("/kaggle/working/results/early_P1/notagtag.json","w"))
'''
open('/tmp/notagtag.py','w').write(test_code)
sh(f"cd {REPO_DIR} && sed -n '/def predict_fold/,/rows.append/p' scripts/evaluate_model.py | head -40")

def predict_fold(args, dataset, test_speaker: str, model_dir: Path) -> list[dict]:
    """Run inference on one fold's test speaker."""
    _, _, test_utts = dataset.loso_split(test_speaker)
    if args.quick > 0:
        test_utts = test_utts[:args.quick]
    print(f"\n[{test_speaker}] {len(test_utts)} test utterances, model {model_dir}")

    predictor = ParaphasiaPredictor(model_dir, device=args.device, num_beams=args.num_beams)

    rows = []
    for i in tqdm(range(0, len(test_utts), args.batch_size), desc=f"Inference {test_speaker}"):
        batch = test_utts[i : i + args.batch_size]
        audios = [
            librosa.load(
                u.audio_path,
                sr=SAMPLING_RATE,
                offset=u.start_time,
                duration=u.end_time - u.start_time,
            )[0]
            for u in batch
        ]
        for utt, hyp in zip(batch, predictor.predict_batch(audios)):
            rows.append({



In [15]:
tc = test_code.replace('''    auds = [librosa.load(u.audio_path, sr=SAMPLING_RATE, offset=u.start_time, duration=u.end_time-u.start_time)[0] if u.end_time else librosa.load(u.audio_path, sr=SAMPLING_RATE)[0] for u in b]
    for u, h in zip(b, pred.predict_batch(auds)):
        rows.append({"reference": " ".join(w + (f" [{l}]" if l != "c" else "") for w, l in zip(u.words, u.labels)), "hypothesis": h})''',
'''    auds = [librosa.load(u.audio_path, sr=SAMPLING_RATE, offset=u.start_time, duration=u.end_time-u.start_time)[0] for u in b]
    for u, h in zip(b, pred.predict_batch(auds)):
        rows.append({"reference": refs[u.utterance_id], "hypothesis": h})''').replace(
'rows = []\n', 'refs = {r["utterance_id"]: r["reference"] for r in json.load(open("/kaggle/working/results/early_P1/predictions.json"))}\nrows = []\n', 1)
open('/tmp/notagtag.py','w').write(tc)
sh(f"cd {REPO_DIR} && CUDA_VISIBLE_DEVICES=0 python /tmp/notagtag.py 2>&1 | grep -v Warning | tail -4")

Recall-[n]: 0.4038
Recall-[s]: 0.6000
Utterances: 91
{'ref[p]': 96, 'hyp[p]': 173, 'ref[n]': 81, 'hyp[n]': 38, 'ref[s]': 34, 'hyp[s]': 128}



In [16]:
sh(f"cd {REPO_DIR} && CUDA_VISIBLE_DEVICES=0 python /tmp/notagtag.py 2>&1 | grep -E 'WER|TD-|F1-' ")
r2 = json.load(open('/kaggle/working/results/early_P1/notagtag.json'))
random.seed(0)
for r in random.sample(r2, 6): print('REF:', r['reference']); print('HYP:', r['hypothesis']); print()

WER:       0.6464
AWER:      0.7338
AWER-PD:   0.6743
TD-binary: 0.9919
TD-[p]:    1.1769
TD-[n]:    1.2074
TD-[s]:    1.7944
TD-all:    4.1788
F1-[p]:    0.7647
F1-[n]:    0.5316
F1-[s]:    0.3846

REF: is easy to use
HYP: then easy [s] is easy [s] is

REF: during the fall it is cool and the leaves change eschuhl [n]
HYP: during the fall it is cool and to leave [s] ang [p] it is cold

REF: pan pan the pambuhld [p] the egg butter and heat heat them up up them over medium heat
HYP: and i [s] may [s] find [p] fall [s] be [s] ay [p] fire [s] in [s] the [s] eezuh [n] pah [n] and have over [s] cold [s] we [p] medium heat [s]

REF: during the fall this [s] is tool [p] and the leaves turn ttuhz [n]
HYP: during the fall this is [s] to [s] and they leave at [s] tern [p] tirtis [n]

REF: fees [p] speak directing to me and din [n] me time for myoonikayt [n]
HYP: please me to write them to me and then me tan [p] for yoonuhkayt [n]

REF: very [s] is ee [n] to use
HYP: feuhli [p] is [s] to use



In [17]:
ana = r'''
import sys, json
sys.path.insert(0, "src")
from collections import Counter
from aphasia_modeling.evaluation.alignment import strip_paraphasia_tags, levenshtein_alignment, EPS

def breakdown(rows):
    err, tot, ins = Counter(), Counter(), 0
    for r in rows:
        rw, rtags = strip_paraphasia_tags(r["reference"].split())
        hw, _ = strip_paraphasia_tags(r["hypothesis"].split())
        i = 0
        for a, b in levenshtein_alignment(rw, hw):
            if a == EPS:
                ins += 1; continue
            cls = rtags.get(i, "c"); i += 1
            tot[cls] += 1; err[cls] += (a != b)
    n = sum(tot.values())
    out = {k: f"{err[k]}/{tot[k]} = {err[k]/tot[k]:.0%}" for k in "cpns"}
    out["insertions"] = ins
    out["WER"] = f"{(sum(err.values())+ins)/n:.1%}"
    out["WER excl. p/n words"] = f"{(err['c']+err['s']+ins)/(tot['c']+tot['s']):.1%} (approx; insertions all charged)"
    return out

for name, path in sys.argv[1:] and [a.split("=") for a in sys.argv[1:]]:
    print(name, json.dumps(breakdown(json.load(open(path))), indent=None))
'''
open('/tmp/ana.py','w').write(ana)
sh(f"cd {REPO_DIR} && python /tmp/ana.py new_P1=/kaggle/working/results/early_P1/notagtag.json")

new_P1 {"c": "233/578 = 40%", "p": "85/96 = 89%", "n": "80/81 = 99%", "s": "23/34 = 68%", "insertions": 89, "WER": "64.6%", "WER excl. p/n words": "56.4% (approx; insertions all charged)"}



In [18]:
zs = r'''
import sys, json, torch, librosa
sys.path.insert(0, "src")
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from aphasia_modeling.data.dataset import AphasiaBankDataset
from aphasia_modeling.model.tokenizer import normalize_output
name = sys.argv[1]
refs = {r["utterance_id"]: r["reference"] for r in json.load(open("/kaggle/working/results/early_P1/predictions.json"))}
_, _, test = AphasiaBankDataset.load("datasets/Scripts/scripts_fridriksson.json").loso_split("P1")
proc = WhisperProcessor.from_pretrained(name)
model = WhisperForConditionalGeneration.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()
rows = []
for i in range(0, len(test), 8):
    b = test[i:i+8]
    auds = [librosa.load(u.audio_path, sr=16000, offset=u.start_time, duration=u.end_time-u.start_time)[0] for u in b]
    feats = proc(auds, sampling_rate=16000, return_tensors="pt").input_features.cuda().half()
    with torch.no_grad():
        out = model.generate(feats, language="en", task="transcribe", max_new_tokens=200)
    for u, t in zip(b, proc.batch_decode(out, skip_special_tokens=True)):
        rows.append({"utterance_id": u.utterance_id, "reference": refs[u.utterance_id], "hypothesis": normalize_output(t)})
json.dump(rows, open(f"/kaggle/working/results/early_P1/zeroshot_{name.split('/')[-1]}.json", "w"))
'''
open('/tmp/zs.py','w').write(zs)
for m in ['openai/whisper-small', 'openai/whisper-large-v3']:
    sh(f"cd {REPO_DIR} && CUDA_VISIBLE_DEVICES=0 python /tmp/zs.py {m} 2>&1 | grep -iE 'error|Traceback' | head -3")
sh(f"cd {REPO_DIR} && python /tmp/ana.py zs_small=/kaggle/working/results/early_P1/zeroshot_whisper-small.json zs_large=/kaggle/working/results/early_P1/zeroshot_whisper-large-v3.json")



zs_small {"c": "369/578 = 64%", "p": "88/96 = 92%", "n": "81/81 = 100%", "s": "28/34 = 82%", "insertions": 428, "WER": "126.0%", "WER excl. p/n words": "134.8% (approx; insertions all charged)"}
zs_large {"c": "317/578 = 55%", "p": "86/96 = 90%", "n": "81/81 = 100%", "s": "25/34 = 74%", "insertions": 185, "WER": "88.0%", "WER excl. p/n words": "86.1% (approx; insertions all charged)"}



In [19]:
zl = {r['utterance_id']: r for r in json.load(open('/kaggle/working/results/early_P1/zeroshot_whisper-large-v3.json'))}
ft = {r['reference']: r['hypothesis'] for r in json.load(open('/kaggle/working/results/early_P1/notagtag.json'))}
random.seed(3)
for uid in random.sample(sorted(zl), 7):
    r = zl[uid]; print('REF:     ', r['reference']); print('large-v3:', r['hypothesis'][:200]); print('ours:    ', ft.get(r['reference'], '?')); print()

REF:      aphasia aphasics [s] me langij [p] not may [p] teleeints [n]
large-v3: aphasia aphasia means ignorance that means tiniest
ours:     aphasia aphasis [s] main anguish [n] not main tidiuhts [n]

REF:      weloh [n] ayz [n] the peestayts [n] is yooyuhli [p] very peasant [p]
large-v3: although the campus free states is usually very busy
ours:     hello [s] aids [s] the [s] have [s] free states is yoo [p] very eduh [n]

REF:      during the tping is [p] is form [p] and sunny
large-v3: the train is his form and setting
ours:     during the tawaym [p] is [p] is form [s] and sunny

REF:      then i choose kip [p] i want [s] like to use
large-v3: i choose to use
ours:     then i choose fit [p] find more like [s] to use

REF:      aphasia affects my language not my intethuhjints [n]
large-v3: my pleasure at that time angry that i
ours:     i [s] uhpuhlt [n] i angwi [n] that i [s]

REF:      during the tumuh [p] is hot with feekwint [p] tunduhstawnz [n]
large-v3: during the summer is hot

In [20]:
PN_RUN = 'scripts-small-pn'
PN_CKPT = f'/root/checkpoints/{PN_RUN}'
os.makedirs(PN_CKPT, exist_ok=True)
cmd = (f'cd {REPO_DIR} && nohup bash -c "while pgrep -f \'run_name {RUN_NAME} \' >/dev/null; do sleep 60; done; '
       f'python scripts/run_loso_parallel.py --gpus 0,1 --data_path {DATA_PATH} --output_dir {PN_CKPT} --run_name {PN_RUN} --wandb_artifacts -- '
       f'--tag_classes pn --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb" >> {PN_CKPT}/runner.log 2>&1 &')
subprocess.run(cmd, shell=True, start_new_session=True)
time.sleep(3)
sh("ps aux | grep -E 'while pgrep|run_loso' | grep -v grep | cut -c1-180")
status()

root         294  0.0  0.4 799212 147476 ?       Sl   00:30   0:01 python3 scripts/run_loso_parallel.py --gpus 0,1 --data_path datasets/Scripts/scripts_fridriksson.json --output_di

[P1] done
[P11] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P11.log
[P10] done
[P12] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P12.log
[P11] done
[P13] training on GPU 0, log: /root/checkpoints/scripts-small-taginit/logs/fold_P13.log
[P12] done
[P2] training on GPU 1, log: /root/checkpoints/scripts-small-taginit/logs/fold_P2.log
---
5
0, 0 %, 7571 MiB
1, 98 %, 7585 MiB

fold_P1.log      epochs= 8 loss=1.359 pred/ref p=133/100 n=29/56 s=22/27 tagacc=0.623  FOLD COMPLETE
fold_P10.log     epochs= 9 loss=1.478 pred/ref p=100/91 n=64/57 s=23/26 tagacc=0.6149  FOLD COMPLETE
fold_P11.log     epochs= 8 loss=1.416 pred/ref p=114/105 n=70/60 s=21/27 tagacc=0.6406  FOLD COMPLETE
fold_P12.log     epochs= 9 loss=1.431 pred/ref p=101/89 n=43/50 s=18/20 tagacc=

In [21]:
sh(f"ps -eo pid,args | grep -v grep | grep -E 'sleep 60|scripts-small-pn' | cut -c1-200; echo; cat {PN_CKPT}/runner.log")

   3671 sleep 60




In [22]:
sh("ps -eo pid,ppid,args | grep -v grep | grep -E 'bash -c|sleep 60' | cut -c1-250")

   3671    3669 sleep 60



In [23]:
sh("ps -o pid,ppid,args -p 3669 | cut -c1-300; ps -eo pid,args | grep -v grep | grep -c 'run_name scripts-small-taginit '")

    PID    PPID COMMAND
   3669       1 bash -c while pgrep -f 'run_name scripts-small-taginit ' >/dev/null; do sleep 60; done; python scripts/run_loso_parallel.py --gpus 0,1 --data_path datasets/Scripts/scripts_fridriksson.json --output_dir /root/checkpoints/scripts-small-pn --run_name scripts-small-pn --wandb_artifacts -
1



In [24]:
sh("kill 3669 3671")
cmd = (f'cd {REPO_DIR} && nohup bash -c "while kill -0 294 2>/dev/null; do sleep 60; done; '
       f'python scripts/run_loso_parallel.py --gpus 0,1 --data_path {DATA_PATH} --output_dir {PN_CKPT} --run_name {PN_RUN} --wandb_artifacts -- '
       f'--tag_classes pn --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb" >> {PN_CKPT}/runner.log 2>&1 &')
subprocess.run(cmd, shell=True, start_new_session=True)
time.sleep(2)
sh("ps -eo pid,ppid,args | grep -v grep | grep -E 'kill -0 294|sleep 60' | cut -c1-120")

/bin/bash: line 1: kill: (3671) - No such process

   3846       1 sleep 60
   3849       1 bash -c while kill -0 294 2>/dev/null; do sleep 60; done; python scripts/run_loso_parallel.py --gpus 0,1
   3850    3849 sleep 60

